# Tune and Train XGBoost Pipeline

This notebook reproduces `tune_and_train.py`: Optuna hyperparameter tuning, model training, SHAP analysis, and model saving. Replace parameters in the configuration cell and run cells sequentially.

In [ ]:
# Imports
import os
import time
import joblib
import optuna
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.pipeline import Pipeline

from pipelines import build_preprocessing_pipeline

# Reduce Optuna verbosity
optuna.logging.set_verbosity(optuna.logging.WARNING)

print('Libraries imported.')

In [ ]:
def print_confusion_matrix(cm, classes):
    """
    Prints a text-based confusion matrix beautifully formatted for command line viewing.
    """
    header = f"{'Actual \\ Pred':<15} | " + " | ".join([f"{cls:<10}" for cls in classes])
    border = "-" * len(header)
    print("\n" + border)
    print(header)
    print(border)
    for i, row in enumerate(cm):
        row_str = f"{classes[i]:<15} | " + " | ".join([f"{val:<10}" for val in row])
        print(row_str)
    print(border + "\n")

In [ ]:
# Notebook configuration — edit these parameters before running
params = {
    'data_path': './phone_tabular_data.csv',
    'n_trials': 30,
    'n_splits': 5,
    'random_state': 42,
    'save_model_path': 'phone_condition_model.joblib',
    'shap_plot_path': 'shap_importance.png'
}

print('Configuration:')
for k, v in params.items():
    print(f"  - {k}: {v}")

In [ ]:
# Load data and prepare train/test split
if not os.path.exists(params['data_path']):
    raise FileNotFoundError(f"Missing tabular dataset file: {params['data_path']}. Please run generate_tabular_data.py first.")

print(f"Loading data from: {os.path.abspath(params['data_path'])}")
df = pd.read_csv(params['data_path'])

X = df.drop(columns=['condition'])
y_raw = df['condition']

classes = ['Reuse', 'Refurbish', 'Repair', 'Recycle']
class_mapping = {cls: idx for idx, cls in enumerate(classes)}
y = y_raw.map(class_mapping)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=params['random_state']
)

print('Data split:')
print('  - Train samples:', X_train.shape[0])
print('  - Test samples:', X_test.shape[0])

In [ ]:
# Optuna objective and tuning
print('\nStarting Optuna hyperparameter tuning...')

def objective(trial):
    params_xgb = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 350),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.25, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 8),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 5.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 5.0, log=True),
        'random_state': params['random_state'],
        'n_jobs': -1,
        'eval_metric': 'mlogloss'
    }
    
    model = xgb.XGBClassifier(**params_xgb)
    pipeline = Pipeline(steps=[
        ('preprocessor', build_preprocessing_pipeline()),
        ('classifier', model)
    ])

    cv = StratifiedKFold(n_splits=params['n_splits'], shuffle=True, random_state=params['random_state'])
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=params['n_trials'])

print('Tuning completed! Best F1-Macro:', study.best_value)
print('\nBest Hyperparameters:')
for k, v in study.best_params.items():
    print(f"  - {k}: {v}")

In [ ]:
# Fit final pipeline on full training set
print('\nFitting final pipeline on entire training set...')
best_params = study.best_params.copy()
best_params['random_state'] = params['random_state']
best_params['n_jobs'] = -1
best_params['eval_metric'] = 'mlogloss'

final_model = xgb.XGBClassifier(**best_params)
final_pipeline = Pipeline(steps=[
    ('preprocessor', build_preprocessing_pipeline()),
    ('classifier', final_model)
])

final_pipeline.fit(X_train, y_train)
print('Final pipeline fitted!')

In [ ]:
# Evaluate on hold-out test set
print('\nEvaluating on test set...')
y_pred = final_pipeline.predict(X_test)

test_acc = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average='macro')

print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"Test F1-Macro: {test_f1:.4f}")

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=classes))

cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print_confusion_matrix(cm, classes)

In [ ]:
# SHAP feature importance
print('\nCalculating SHAP feature importance...')
preprocessor = final_pipeline.named_steps['preprocessor']
X_test_preprocessed = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()
clean_feature_names = [name.split('__')[1] if '__' in name else name for name in feature_names]
X_test_df = pd.DataFrame(X_test_preprocessed, columns=clean_feature_names)

classifier = final_pipeline.named_steps['classifier']
explainer = shap.TreeExplainer(classifier)
shap_values = explainer.shap_values(X_test_df)

print('Generating SHAP summary plot...')
plt.figure(figsize=(12, 8))
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
shap.summary_plot(shap_values, X_test_df, plot_type='bar', class_names=classes, show=False)
plt.title('SHAP Feature Importance (Condition Prediction Model)')
plt.tight_layout()
plt.savefig(params['shap_plot_path'], dpi=200)
plt.close()
print(f"SHAP plot saved to: {os.path.abspath(params['shap_plot_path'])}")

In [ ]:
# Save model and verify inference
joblib.dump(final_pipeline, params['save_model_path'])
print(f"Model saved to: {os.path.abspath(params['save_model_path'])}")

print('\nVerifying inference on a simulated new record...')
loaded = joblib.load(params['save_model_path'])
raw_sample = pd.DataFrame([{
    'brand': 'Apple',
    'model': 'iPhone 13',
    'age_years': 1.2,
    'battery_health_pct': 87,
    'screen_condition': 3,
    'body_condition': 2,
    'functional_issues': 'battery',
    'water_damage_history': 0,
    'repair_history': 0,
    'original_purchase_price': 899.0
}])

pred_idx = loaded.predict(raw_sample)[0]
pred_probs = loaded.predict_proba(raw_sample)[0]
print('\nPredicted class index:', pred_idx)
print('Predicted label:', classes[pred_idx])
print('Class probabilities:')
for i, cname in enumerate(classes):
    print(f"  - {cname}: {pred_probs[i]*100:.2f}%")